# Bonus: Running the Live Digital Twin

*Optional, after class. Requires Docker, about 15 minutes.*

OpenDT is a Docker Compose microservice system in which the core components (data source, simulator, calibrator, and
API) communicate through Apache Kafka; a Grafana dashboard shows the results.
In this notebook, we (1) start the live digital twin of SURF, (2) query its predictions from Python, (3) change the
hosts from the REST API, and (4) run it with the Self-Calibrator.

Run the `make` commands in a terminal, in the OpenDT folder (the folder with the `Makefile`), after the setup in the
README.

## 1. Starting the Live Twin

```
make up config=tutorial/configs/live_twin.yaml
```

`make up` deletes the data of the previous run (the Kafka and Grafana volumes) and starts all services; the first start
builds the Docker images, which takes a few minutes.
The configuration replays SURF at maximum speed (`speed_factor: -1`), with windows of operation of 5 minutes.

Then, open the dashboard at http://localhost:3000 ("OpenDT Dashboard").
It shows the ground truth and the prediction of OpenDT for 6-9 October 2022; to see later days, change the time range
at the top right.

## 2. Predictions from the API

OpenDT also serves its predictions through a REST API (documentation at http://localhost:3001/docs).
The next cell asks for the power draw, in 5-minute samples.
Until OpenDT finishes its first window of operation, the API answers with an error (HTTP 404); in this case, wait a
minute and run the cell again.

In [ ]:
import json
import urllib.request

import pandas as pd
from opendt_tutorial import PATHS, legend_above

API = "http://localhost:3001"

with urllib.request.urlopen(f"{API}/api/power?interval_seconds=300") as response:
    power = pd.DataFrame(json.load(response)["data"])
power = power.set_index(pd.to_datetime(power["timestamp"]))
power = power.rename(columns={"actual_power": "Ground Truth", "simulated_power": "OpenDT"})

ax = (power[["Ground Truth", "OpenDT"]] / 1000).plot(figsize=(10, 4), color=["#999999", "#009E73"])
ax.set(xlabel="Time [UTC]", ylabel="Power Draw [kW]", ylim=(0, None))
legend_above(ax)

## 3. Changing the Hosts from the REST API

As in the live demo, we send OpenDT a topology with 10x the hosts: `tutorial/configs/topology_2770_hosts.json` is the
SURF topology of OpenDT, with 2,770 hosts instead of 277.

*Question:* what do we expect to see on the dashboard after the change? Write down a guess, then run the next cell and
watch the dashboard.

In [ ]:
body = (PATHS.root / "tutorial/configs/topology_2770_hosts.json").read_bytes()
request = urllib.request.Request(
    f"{API}/api/topology", data=body, method="PUT", headers={"Content-Type": "application/json"}
)
with urllib.request.urlopen(request) as response:
    print(json.load(response))

<details><summary>Answer</summary>

From the next window of operation, OpenDT predicts the datacenter with 2,770 hosts: the predicted power draw
jumps to about 4x, while the ground truth does not change. Notebook 1 explains why.
</details>

## 4. The Self-Calibrator

We stop OpenDT and start it again with the Self-Calibrator (the grid search of notebook 2):

```
make down
make up config=tutorial/configs/live_twin_calibration.yaml
make logs-calibrator
```

After each grid search, the calibrator logs the MAPE of each candidate value of r, the best value
(`Best: cpuPowerModel.calibrationFactor=...`), and publishes it to the simulator (`Published topology with ...`).
Compare the values it picks with the best r per day in notebook 2. Stop following the logs with Ctrl+C.

## 5. Stopping the Live Twin

```
make down
```

The data of each run (inputs, outputs, and results of OpenDT) stays in `data/<run id>/`.

## Summary

In this notebook, we (1) start the live digital twin of SURF, (2) query its predictions from the REST API, (3) change
the hosts from the REST API, and (4) run it with the Self-Calibrator.

*Learn more:* [prerequisites](https://atlarge-research.github.io/opendt/docs/getting-started/prerequisites) (Docker resources),
[running OpenDT](https://atlarge-research.github.io/opendt/docs/getting-started/running), the [API](https://atlarge-research.github.io/opendt/docs/services/api), and the
[OpenDT paper](https://atlarge-research.com/pdfs/2026-icpe-opendt.pdf).